In [ ]:
import sys
import gc
from pathlib import Path

import torch

sys.path.append("./Efficient_Architecture")

from src.utils.models import Qwen3, Qwen35, LFM2, IBM_Granite1b, IBM_Granite, GPT2
from data.preprocessing import c4_dataset
from src.test import test_model, test_suite
from src.utils.visuals import generate_plots
from src.utils.metrics import non_embedding_params, count_params

import json

# Choose dataset configuration
LANGUAGE = "en"  # e.g. "en"
SPLIT = "train"  # e.g. "train"

# Write all outputs here (helps a lot on Colab where CWD can be surprising)
OUT_DIR = Path("metrics")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Model name -> class (instantiate one at a time to limit memory)
model_classes = {
    "Qwen3": Qwen3,
    "Qwen3.5": Qwen35,
    "LFM2": LFM2,
    "IBM-G1B": IBM_Granite1b,
    "IBM-G350M": IBM_Granite,
    "GPT2": GPT2,
}

results_files = []

for name, ModelClass in model_classes.items():
    print(f"\n=== {name} ===")

    wrapper = None
    dataset = None
    metrics = None
    json_path = OUT_DIR / f"{name}_metrics.json"

    try:
        wrapper = ModelClass()
        total_p = count_params(wrapper.model)
        non_emb_p = non_embedding_params(wrapper.model)
        print(
            f"Parameter count: total={total_p:,} | non-embedding={non_emb_p:,} | embedding+head={total_p - non_emb_p:,}"
        )

        print(f"Running metrics for {name}...")
        dataset = c4_dataset(split=SPLIT, language=LANGUAGE, tokenizer=wrapper.tokenizer)
        metrics = test_model(wrapper.model, dataset)

        with open(json_path, "w") as f:
            serializable = {}
            for (read_len, gen_len), vals in metrics.items():
                serializable[f"({read_len},{gen_len})"] = vals
            json.dump(serializable, f)

        results_files.append(str(json_path))
        print(f"Saved metrics to {json_path}\n")

    except torch.cuda.OutOfMemoryError as e:
        err = str(e)
        print(f"CUDA OOM for {name}. Writing empty results and skipping...")
        with open(json_path, "w") as f:
            serializable = {f"({read_len},{gen_len})": {} for (read_len, gen_len) in test_suite}
            serializable["_status"] = "oom"
            serializable["error"] = err
            json.dump(serializable, f)

    except RuntimeError as e:
        msg = str(e).lower()
        if "out of memory" in msg or "cuda" in msg and "memory" in msg:
            print(f"RuntimeError OOM for {name}. Writing empty results and skipping...")
            with open(json_path, "w") as f:
                serializable = {f"({read_len},{gen_len})": {} for (read_len, gen_len) in test_suite}
                serializable["_status"] = "oom"
                serializable["error"] = str(e)
                json.dump(serializable, f)
        else:
            raise

    finally:
        # Free memory before loading next model
        if metrics is not None:
            del metrics
        if dataset is not None:
            del dataset
        if wrapper is not None:
            del wrapper

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# Generate comparison plots
if results_files:
    print("Generating plots...")
    generate_plots(results_files, str(OUT_DIR / "plots"))
    print(f"Plots saved as {OUT_DIR / 'plots.png'}")
